# Deployment-Friendly Model Artifact

This notebook creates a deployment-friendly saved model artifact for the Board Game Rating Predictor project.

The previous model improvement notebook selected Random Forest as the best-performing model.

However, the first saved Random Forest `.joblib` file was too large for normal GitHub storage.

This notebook tests a smaller and/or compressed model artifact so the project can later be used more easily in a Streamlit dashboard and deployment environment.

In [36]:
# Import libraries uesd throghout this notebook
# joblib saves models; pandas loads CSV files; numpy helps calculate RMSE
import joblib
import numpy as np
import pandas as pd

# Import Path for reliable file paths across operating systems
from pathlib import Path

# Import modelling and evaluation tools
# RandomForestRegressor = to traoin smaller version of current best model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [37]:
# for defineing current notebook folder and project root
current_folder = Path.cwd()
project_root = current_folder.parent

print("Current folder:", current_folder)
print("Project root:", project_root)

Current folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\notebooks
Project root: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor


## Load Prepared Modelling Data

The same prepared training and testing datasets are loaded.

Using the same data allows the deployment-friendly model to be compared fairly with the previously selected Random Forest model.

In [38]:
# to define important project folders
processed_data_folder = project_root / "data" / "processed"
models_folder = project_root / "models"

print("Processed data folder:", processed_data_folder)
print("Processed data folder exists:", processed_data_folder.exists())

print("Models folder:", models_folder)
print("Models folder exists:", models_folder.exists())

Processed data folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed
Processed data folder exists: True
Models folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\models
Models folder exists: True


In [39]:
# To define paths to prepared modelling files
x_train_path = processed_data_folder / "x_train_prepared.csv"
x_test_path = processed_data_folder / "x_test_prepared.csv"
y_train_path = processed_data_folder / "y_train.csv"
y_test_path = processed_data_folder / "y_test.csv"
model_feature_names_path = processed_data_folder / "model_feature_names.csv"
best_model_summary_path = processed_data_folder / "best_model_summary.csv"

required_paths = [
    x_train_path,
    x_test_path,
    y_train_path,
    y_test_path,
    model_feature_names_path,
    best_model_summary_path
]

for path in required_paths:
    print(path.name, "exists:", path.exists())

x_train_prepared.csv exists: True
x_test_prepared.csv exists: True
y_train.csv exists: True
y_test.csv exists: True
model_feature_names.csv exists: True
best_model_summary.csv exists: True


In [40]:
# Load same previously used train/test data
X_train = pd.read_csv(x_train_path)
X_test = pd.read_csv(x_test_path)

y_train_data = pd.read_csv(y_train_path)
y_test_data = pd.read_csv(y_test_path)

model_feature_names = pd.read_csv(model_feature_names_path)
best_model_summary = pd.read_csv(best_model_summary_path)

target_column = "Rating Average"

y_train = y_train_data[target_column]
y_test = y_test_data[target_column]

print("Prepared modelling datasets loaded successfully.")

Prepared modelling datasets loaded successfully.


In [41]:
# To confirm we are using same modelling data adn 
# previous model summary loaded correctly
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("model_feature_names shape:", model_feature_names.shape)
print("best_model_summary shape:", best_model_summary.shape)

X_train shape: (16273, 35)
X_test shape: (4069, 35)
y_train shape: (16273,)
y_test shape: (4069,)
model_feature_names shape: (35, 1)
best_model_summary shape: (1, 7)


In [42]:
# Verify that loaded feature columns match the saved model feature names
expected_feature_names = model_feature_names["Feature"].tolist()

x_train_columns_match = X_train.columns.tolist() == expected_feature_names
x_test_columns_match = X_test.columns.tolist() == expected_feature_names

print("X_train columns match saved feature names:", x_train_columns_match)
print("X_test columns match saved feature names:", x_test_columns_match)

X_train columns match saved feature names: True
X_test columns match saved feature names: True


In [43]:
# Review previous best model summary
# -> to compare how close it is to previous best model
best_model_summary

,Best Model,Test MAE,Test RMSE,Test R2,Model File,Number of Features,Selection Reason
0,Random Forest,0.491151,0.664943,0.499325,models\random_forest_rating_model.joblib,35,"Random Forest had the best test-set MAE, RMSE,..."


## Helper Functions

Reusable helper functions are created for model evaluation and file-size checking.

These helpers make it easier to compare candidate deployment-friendly model artifacts.

In [44]:
# Create a reusable function for evaluating regression model performance
def evaluate_regression_model(y_true, y_pred):
    """Calculate common regression evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    }

In [45]:
# Create reusable function to check file size (megabytes)
def get_file_size_mb(file_path):
    """Return file size in megabytes."""
    file_size_bytes = file_path.stat().st_size
    file_size_mb = file_size_bytes / (1024 * 1024)

    return file_size_mb

### Deployment-Friendly Artifact Setup Result

The prepared modelling datasets were loaded successfully.

The feature columns were verified against the saved model feature list.

The previous best-model summary was loaded for comparison.

Reusable helper functions were created for model evaluation and file-size checking.

The notebook is ready to test deployment-friendly model artifact options.

## Test Compressed Random Forest Artifact

The first deployment-friendly artifact option uses the same Random Forest settings as the selected best model, but saves the model with joblib compression.

This checks whether compression alone can reduce the saved model file size enough for GitHub and deployment.

If compression works, we may be able to keep the same model performance while producing a smaller saved artifact.

In [46]:
# Train Random Forest candidate using same settings as the selected best model
# -> model should match previous best model settings
compressed_random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

compressed_random_forest_model.fit(X_train, y_train)

print("Compressed Random Forest candidate trained successfully.")
print("Number of trees:", compressed_random_forest_model.n_estimators)
print("Number of input features:", compressed_random_forest_model.n_features_in_)

Compressed Random Forest candidate trained successfully.
Number of trees: 100
Number of input features: 35


In [47]:
# Evaluate compressed Random Forest candidate
compressed_random_forest_test_predictions = compressed_random_forest_model.predict(X_test)

compressed_random_forest_test_metrics = evaluate_regression_model(
    y_test,
    compressed_random_forest_test_predictions
)

print("Compressed Random Forest test metrics:", compressed_random_forest_test_metrics)

Compressed Random Forest test metrics: {'MAE': 0.4911510570047631, 'RMSE': 0.6649429553038372, 'R2': 0.4993247012615084}


In [48]:
# Save Random Forest candidate with joblib compression
# -> tells joblib to compress the saved model file
# -> might come with the price of saving & loading takes time, but size is much smaller
compressed_random_forest_path = (
    models_folder / "compressed_random_forest_rating_model.joblib"
)

joblib.dump(
    compressed_random_forest_model,
    compressed_random_forest_path,
    compress=3
)

compressed_random_forest_size_mb = get_file_size_mb(
    compressed_random_forest_path
)

print("Compressed Random Forest model saved successfully.")
print("File exists:", compressed_random_forest_path.exists())
print("Compressed Random Forest file size MB:", compressed_random_forest_size_mb)

Compressed Random Forest model saved successfully.
File exists: True
Compressed Random Forest file size MB: 27.314017295837402


In [49]:
# Check whether compressed model file is small enough for normal GitHub saving
github_file_size_limit_mb = 100

compressed_random_forest_under_github_limit = (
    compressed_random_forest_size_mb < github_file_size_limit_mb
)

print(
    "Compressed Random Forest under GitHub 100 MB limit:",
    compressed_random_forest_under_github_limit
)

print(
    "Size margin below/above limit MB:",
    github_file_size_limit_mb - compressed_random_forest_size_mb
)

Compressed Random Forest under GitHub 100 MB limit: True
Size margin below/above limit MB: 72.6859827041626


In [50]:
# Compare compressed Random Forest candidate with the previous best 
# model summary in comparison table
previous_best_test_mae = best_model_summary.loc[0, "Test MAE"]
previous_best_test_rmse = best_model_summary.loc[0, "Test RMSE"]
previous_best_test_r2 = best_model_summary.loc[0, "Test R2"]

compressed_candidate_comparison = pd.DataFrame(
    [
        {
            "Model Artifact": "Previous Best Random Forest",
            "Test MAE": previous_best_test_mae,
            "Test RMSE": previous_best_test_rmse,
            "Test R2": previous_best_test_r2,
            "File Size MB": np.nan,
            "Under GitHub 100 MB Limit": False
        },
        {
            "Model Artifact": "Compressed Random Forest",
            "Test MAE": compressed_random_forest_test_metrics["MAE"],
            "Test RMSE": compressed_random_forest_test_metrics["RMSE"],
            "Test R2": compressed_random_forest_test_metrics["R2"],
            "File Size MB": compressed_random_forest_size_mb,
            "Under GitHub 100 MB Limit": compressed_random_forest_under_github_limit
        }
    ]
)

compressed_candidate_comparison_rounded = compressed_candidate_comparison.copy()

compressed_candidate_comparison_rounded[
    ["Test MAE", "Test RMSE", "Test R2", "File Size MB"]
] = (
    compressed_candidate_comparison_rounded[
        ["Test MAE", "Test RMSE", "Test R2", "File Size MB"]
    ]
    .round(4)
)

compressed_candidate_comparison_rounded

,Model Artifact,Test MAE,Test RMSE,Test R2,File Size MB,Under GitHub 100 MB Limit
0,Previous Best Random Forest,0.4912,0.6649,0.4993,NaN,False
1,Compressed Random Forest,0.4912,0.6649,0.4993,27.314,True


### Compressed Random Forest Artifact Test Result

A Random Forest model with the same settings as the selected best model was trained and saved with joblib compression.

The model was evaluated on the test set so its performance could be compared with the previously selected best model.

The compressed file size was checked against GitHub's normal 100 MB file size limit.

If the compressed artifact is still too large, a smaller Random Forest candidate will be tested next.

## Select Deployment-Friendly Model Artifact

The compressed Random Forest model is selected as the deployment-friendly model artifact.

Compression reduced the saved model file size below GitHub's normal 100 MB file limit while keeping the same test-set performance as the previously selected Random Forest model.

This artifact is suitable for later use in the Streamlit dashboard because it can be loaded directly without retraining the model.

In [51]:
# Select compressed Random Forest as the deployment-friendly model artifact
# -> crates variable names for final selected deployment artifact
selected_deployment_model_name = "Compressed Random Forest"
selected_deployment_model = compressed_random_forest_model
selected_deployment_model_path = compressed_random_forest_path
selected_deployment_model_size_mb = compressed_random_forest_size_mb
selected_deployment_model_metrics = compressed_random_forest_test_metrics

print("Selected deployment model:", selected_deployment_model_name)
print(
    "Selected deployment model path:",
    selected_deployment_model_path.relative_to(project_root)
)
print("Selected deployment model file size MB:", selected_deployment_model_size_mb)
print(
    "Under GitHub 100 MB limit:",
    compressed_random_forest_under_github_limit
)

Selected deployment model: Compressed Random Forest
Selected deployment model path: models\compressed_random_forest_rating_model.joblib
Selected deployment model file size MB: 27.314017295837402
Under GitHub 100 MB limit: True


In [52]:
# Load selected deployment-friendly model artifact as a safety check
# -> to confirm it is not only acceptabel size, but usable
loaded_deployment_model = joblib.load(selected_deployment_model_path)

print("Deployment-friendly model loaded successfully.")
print("Loaded model type:", type(loaded_deployment_model))
print("Loaded model number of features:", loaded_deployment_model.n_features_in_)

Deployment-friendly model loaded successfully.
Loaded model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Loaded model number of features: 35


In [53]:
# Comprare predictions from the selected model and the loaded deployment model
selected_model_sample_predictions = selected_deployment_model.predict(
    X_test.head(5)
)

loaded_deployment_model_sample_predictions = loaded_deployment_model.predict(
    X_test.head(5)
)

deployment_prediction_check = pd.DataFrame(
    {
        "Selected Model Prediction": selected_model_sample_predictions,
        "Loaded Deployment Model Prediction": (
            loaded_deployment_model_sample_predictions
        )
    }
)

deployment_prediction_check["Absolute Difference"] = (
    deployment_prediction_check["Selected Model Prediction"]
    - deployment_prediction_check["Loaded Deployment Model Prediction"]
).abs()

deployment_prediction_check["Predictions Match Within Tolerance"] = np.isclose(
    deployment_prediction_check["Selected Model Prediction"],
    deployment_prediction_check["Loaded Deployment Model Prediction"],
    rtol=1e-10,
    atol=1e-10
)

print(
    "All deployment predictions match within tolerance:",
    deployment_prediction_check[
        "Predictions Match Within Tolerance"
    ].all()
)

print(
    "Maximum absolute difference:",
    deployment_prediction_check["Absolute Difference"].max()
)

deployment_prediction_check

All deployment predictions match within tolerance: True
Maximum absolute difference: 1.7763568394002505e-15


,Selected Model Prediction,Loaded Deployment Model Prediction,Absolute Difference,Predictions Match Within Tolerance
0,6.5242,6.5242,0.000000e+00,True
1,6.4193,6.4193,1.776357e-15,True
2,7.1308,7.1308,0.000000e+00,True
3,6.4677,6.4677,0.000000e+00,True
4,7.6670,7.6670,0.000000e+00,True


In [54]:
# Save the summary of the selected deployment-friendly model artifact
deployment_friendly_model_artifact_summary = pd.DataFrame(
    [
        {
            "Selected Deployment Model": selected_deployment_model_name,
            "Model File": str(
                selected_deployment_model_path.relative_to(project_root)
            ),
            "Test MAE": selected_deployment_model_metrics["MAE"],
            "Test RMSE": selected_deployment_model_metrics["RMSE"],
            "Test R2": selected_deployment_model_metrics["R2"],
            "File Size MB": selected_deployment_model_size_mb,
            "GitHub File Size Limit MB": github_file_size_limit_mb,
            "Under GitHub Limit": compressed_random_forest_under_github_limit,
            "Number of Features": selected_deployment_model.n_features_in_,
            "Selection Reason": (
                "Joblib compression reduced the Random Forest model file "
                "below GitHub's normal 100 MB limit while preserving the "
                "same test-set performance."
            )
        }
    ]
)

deployment_friendly_model_artifact_summary_path = (
    processed_data_folder / "deployment_friendly_model_artifact_summary.csv"
)

deployment_friendly_model_artifact_summary.to_csv(
    deployment_friendly_model_artifact_summary_path,
    index=False
)

print("Deployment-friendly model artifact summary saved successfully.")
print("File exists:", deployment_friendly_model_artifact_summary_path.exists())

deployment_friendly_model_artifact_summary

Deployment-friendly model artifact summary saved successfully.
File exists: True


,Selected Deployment Model,Model File,Test MAE,Test RMSE,Test R2,File Size MB,GitHub File Size Limit MB,Under GitHub Limit,Number of Features,Selection Reason
0,Compressed Random Forest,models\compressed_random_forest_rating_model.j...,0.491151,0.664943,0.499325,27.314017,100,True,35,Joblib compression reduced the Random Forest m...


In [55]:
# Create the final conclusion for the deployment-friendly model artifact
deployment_artifact_conclusion = (
    f"The selected deployment-friendly model artifact is "
    f"{selected_deployment_model_name}. "
    f"It was saved to "
    f"{selected_deployment_model_path.relative_to(project_root)}. "
    f"The compressed file size is {selected_deployment_model_size_mb:.2f} MB, "
    f"which is below GitHub's normal {github_file_size_limit_mb} MB file size "
    f"limit. The model preserved the selected Random Forest test performance "
    f"with MAE of {selected_deployment_model_metrics['MAE']:.4f}, "
    f"RMSE of {selected_deployment_model_metrics['RMSE']:.4f}, and "
    f"R2 of {selected_deployment_model_metrics['R2']:.4f}. "
    "The compressed model was loaded back successfully and produced matching "
    "predictions within floating-point tolerance."
)

deployment_artifact_conclusion

"The selected deployment-friendly model artifact is Compressed Random Forest. It was saved to models\\compressed_random_forest_rating_model.joblib. The compressed file size is 27.31 MB, which is below GitHub's normal 100 MB file size limit. The model preserved the selected Random Forest test performance with MAE of 0.4912, RMSE of 0.6649, and R2 of 0.4993. The compressed model was loaded back successfully and produced matching predictions within floating-point tolerance."

### Deployment-Friendly Model Artifact Result

The compressed Random Forest model was selected as the deployment-friendly model artifact.

The compressed model preserved the same test-set performance as the selected Random Forest model.

The saved compressed artifact is below GitHub's normal 100 MB file size limit.

The compressed model was loaded back successfully and produced matching predictions within floating-point tolerance.

A deployment-friendly model artifact summary was saved in the `data/processed` folder.

This artifact can be used later by the Streamlit dashboard without retraining the model each time.